# Summary

Test the Google ADK evaluation tools

In [34]:
import os, sys
import json
import time

import pandas as pd

import vertexai
from google.adk.evaluation import AgentEvaluator

from vertexai import agent_engines

from vertexai.evaluation import (
    EvalTask,
    MetricPromptTemplateExamples,
    PairwiseMetric,
    PointwiseMetric,
    PointwiseMetricPromptTemplate,
)




utils_path = "../interface/utils"
sys.path.insert(0, utils_path)
from authentication import ApiAuthentication

# Set environment variables
dotenv_path = "../data/environment"
api_configs = ApiAuthentication(dotenv_path=dotenv_path)
api_configs.set_environ_variables()

# Initialize Vertex AI API once per session
vertexai.init(project=os.environ["GOOGLE_CLOUD_PROJECT"],
              location=os.environ["GOOGLE_CLOUD_LOCATION"],
              staging_bucket=os.environ["STAGING_BUCKET"])


## Create a test file in a Google ADK evaluation or Vertex AI format - Vertex AI formats can vary depending on test


In [2]:
qa_adk_data_path = "eval/data"
qa_data_path = "../evaluation/rag_tests/data/eval_data"
# os.listdir(qa_data_path)

qa_file = "ipeds_qa_1.json"

# Read in q and a format
with open(os.path.join(qa_data_path, qa_file), "r") as file:
    qa_base = json.load(file)

# qa_base

# dir(qa_base)

# Create a dataset in a format for Vertex AI Evaluation
prompts = [qa[0] for qa in qa_base]
responses = [qa[1] for qa in qa_base]

vai_eval_dataset = pd.DataFrame({"prompt": prompts,
                                 "baseline_model_response": responses,
                                 })




In [3]:
qas_adk_list
vai_eval_dataset

# instruction_prompt_v1 = """
#         You are an AI assistant with access to specialized corpus of documents.
#         Your role is to provide accurate and concise answers to questions based
#         on documents that are retrievable using ask_vertex_retrieval. If you believe
#         the user is just chatting and having casual conversation, don't use the retrieval tool.
#
#         But if the user is asking a specific question about a knowledge they expect you to have,
#         you can use the retrieval tool to fetch the most relevant information.
#
#         If you are not certain about the user intent, make sure to ask clarifying questions
#         before answering. Once you have the information you need, you can use the retrieval tool
#         If you cannot provide an answer, clearly explain why.
#         """
#
# vai_eval_dataset["system_prompt"] = instruction_prompt_v1
# vai_eval_dataset = vai_eval_dataset.rename(columns={"prompt": "question", "baseline_model_response": "reference"})
# vai_eval_dataset


# vai_eval_dataset.loc[0, "system_prompt"]


,prompt,baseline_model_response
0,Which California community college awarded the...,"Using the most recent IPEDS data, WEST LOS ANG..."
1,Which California community college has the hig...,"Using the most recent IPEDS data, MT SAN ANTON..."
2,Which California community college has the hig...,"Using the most recent IPEDS data, EAST LOS ANG..."
3,Which California community college has the hig...,"Using the most recent IPEDS data, EVERGREEN VA..."
4,Which California community college has the low...,"Using the most recent IPEDS data, SHASTA COLLE..."
5,Which California community college has the hig...,"Using the most recent IPEDS data, SANTA MONICA..."
6,Which California community college has the low...,"Using the most recent IPEDS data, SOLANO COMMU..."
7,Which California community college has the mos...,"Using the most recent IPEDS data, SOUTHWESTERN..."
8,Which California community college had the mos...,"Using the most recent IPEDS data, ORANGE COAST..."
9,How many California community college undergra...,"Using the most recent IPEDS data, 1491.0 under..."


## Run tests

In [7]:
from vertexai.evaluation import EvalTask
import asyncio

from vertexai.evaluation import (
    EvalTask,
    MetricPromptTemplateExamples,
    PairwiseMetric,
    PointwiseMetric,
    PointwiseMetricPromptTemplate,
)

from vertexai import agent_engines

# Retrieve an ADK agent
agent_engine = agent_engines.get(os.getenv("AGENT_ENGINE_ID"))








In [9]:
# from vertexai.generative_models import GenerativeModel
#
# model = GenerativeModel(model_name="gemini-pro")
#
# rouge_eval_task = EvalTask(
#     dataset=vai_eval_dataset,
#     metrics=["rouge_l_sum"],
# )
# rouge_result = rouge_eval_task.evaluate(
#     model=agent_engine,
#     prompt_template="# System_prompt\n{system_prompt} # Question\n{question}",
# )

In [11]:

# def test_with_single_test_file():
#     """Test the agent's basic ability via a session file."""
#     AgentEvaluator.evaluate(
#         agent_module="rag",
#         eval_dataset_file_path_or_dir=os.path.join(qa_adk_data_path, qa_adk_file),
#     )
#
# test_class = AgentEvaluator()

## Test using ADK evaluation tools

In [26]:
qa_adk_data_path = "eval/data"
qa_data_path = "../evaluation/rag_tests/data/eval_data"
# os.listdir(qa_data_path)

qa_file = "ipeds_qa_1.json"

# Read in q and a format
with open(os.path.join(qa_data_path, qa_file), "r") as file:
    qa_base = json.load(file)

# Create a list of dictionaries in ADK format
qas_adk_list = []
for qa in qa_base:
    qas_adk_list.append(dict(query=qa[0],
                             expected_tool_use=[],
                             expected_intermediate_agent_responses=[],
                             reference=qa[1]))

# Write a file in the ADK format
qa_adk_file = "ccc_test.test.json"
# with open(os.path.join(qa_adk_data_path, qa_adk_file), "w") as file:
#     file.write(json.dumps(qas_adk_list))



###  Run the evaluation in a cell and capture screen output using Jupyter Magic command

In [27]:
%%capture cap


# Run tests
test = AgentEvaluator.evaluate(agent_module="rag",
                               eval_dataset_file_path_or_dir=os.path.join(qa_adk_data_path, qa_adk_file),
                               )

# Save the captured output to a text file
if 'cap' in locals():
    with open(os.path.join(qa_adk_data_path, "test_results.txt"), "w") as file:
        file.write(cap.stdout)


Associating projects/1062597788108/locations/us-central1/metadataStores/default/contexts/test-1-fb34c119-3f66-4b3f-ab64-d9d0ac3e5099 to Experiment: test-1
Computing metrics with a total of 40 Vertex Gen AI Evaluation Service API requests.
All 40 metric requests are successfully computed.
Evaluation Took:4.151141250040382 seconds


In [29]:
# read test results
with open(os.path.join(qa_adk_data_path, "test_results.txt"), "r") as file:
    results_txt = file.read()

results_txt[:1000]

"Evaluation Summary Metrics: {'row_count': 40, 'rouge_1/mean': np.float64(0.28844972474999997), 'rouge_1/std': np.float64(0.13643181691356795)}\n+----+------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------+-----------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Test using Vertex AI tools

### Create an evaluation dataset

In [31]:
qa_adk_data_path = "eval/data"
qa_data_path = "../evaluation/rag_tests/data/eval_data"
# os.listdir(qa_data_path)

qa_file = "ipeds_qa_1.json"

# Read in q and a format
with open(os.path.join(qa_data_path, qa_file), "r") as file:
    qa_base = json.load(file)

# Create a dataset in a format for Vertex AI Evaluation
prompts = [qa[0] for qa in qa_base]
responses = [qa[1] for qa in qa_base]

# Create a dataframe
vai_eval_dataset = pd.DataFrame({"prompt": prompts,
                                 "baseline_model_response": responses,
                                 })

vai_eval_dataset.head()


,prompt,baseline_model_response
0,Which California community college awarded the...,"Using the most recent IPEDS data, WEST LOS ANG..."
1,Which California community college has the hig...,"Using the most recent IPEDS data, MT SAN ANTON..."
2,Which California community college has the hig...,"Using the most recent IPEDS data, EAST LOS ANG..."
3,Which California community college has the hig...,"Using the most recent IPEDS data, EVERGREEN VA..."
4,Which California community college has the low...,"Using the most recent IPEDS data, SHASTA COLLE..."


## Add agent responses

In [35]:
# Function to extract agent response
def pretty_event_response(event):
    """Pretty prints an event with truncation for long content."""
    if "content" not in event:
        return None

    parts = event["content"].get("parts", [])

    for part in parts:
        if "text" in part:
            text = part["text"]
            return text

    return None

# Retreive an ADK agent
agent_engine = agent_engines.get(os.getenv("AGENT_ENGINE_ID"))

# Create a session
user_id = "u_123"
session = agent_engine.create_session(user_id=user_id)

# Create a model function to return agent's response
def test_model(user_id,
               session_id,
               query):

    # Look for events for each query
    text = None
    for event in agent_engine.stream_query(user_id=user_id,
                                           # session_id=session["id"],
                                           session_id=session_id,
                                           message=query,):

        text = pretty_event_response(event)
        if text:
            break

    return text

# Add model responses to Q and A dataframe
vai_eval_dataset["response"] = ""
# for idx in vai_eval_dataset.index:
#     # print(idx)
#     # print(vai_eval_dataset.loc[idx, "prompt"])
#     vai_eval_dataset.loc[idx, "response"] = test_model(user_id=user_id,
#                                                        session_id=session["id"],
#                                                        query=vai_eval_dataset.loc[idx, "prompt"])
#
#     # Hitting quota limits
#     # Error: 429 Quota exceeded for quota metric 'Query Reasoning Engine requests' and limit 'Query Reasoning Engine requests per minute per region'
#     time.sleep(10)



In [41]:
# save as CSV for use later
# vai_eval_dataset.head()
# vai_eval_dataset.to_csv(os.path.join(qa_adk_data_path, "vai_eval_dataset.csv"), index=False)

# Read test dataset
vai_eval_dataset_2 = pd.read_csv(os.path.join(qa_adk_data_path, "vai_eval_dataset.csv"))
vai_eval_dataset_2.head()


,prompt,baseline_model_response,response
0,Which California community college awarded the...,"Using the most recent IPEDS data, WEST LOS ANG...","I am sorry, but based on the provided document..."
1,Which California community college has the hig...,"Using the most recent IPEDS data, MT SAN ANTON...","I am sorry, but the provided documents do not ..."
2,Which California community college has the hig...,"Using the most recent IPEDS data, EAST LOS ANG...","I am sorry, but the provided documents do not ..."
3,Which California community college has the hig...,"Using the most recent IPEDS data, EVERGREEN VA...","As of Fall 2006, salaries for tenured and tenu..."
4,Which California community college has the low...,"Using the most recent IPEDS data, SHASTA COLLE...","I am sorry, but the provided documents do not ..."


## Use VertexAI to conduct a rouge_l_sum test

In [44]:
# rename columns in test dataframe
vai_eval_dataset_3 = vai_eval_dataset_2.rename(columns={"prompt": "instruction", "baseline_model_response": "reference"})
vai_eval_dataset_3.head()


,instruction,reference,response
0,Which California community college awarded the...,"Using the most recent IPEDS data, WEST LOS ANG...","I am sorry, but based on the provided document..."
1,Which California community college has the hig...,"Using the most recent IPEDS data, MT SAN ANTON...","I am sorry, but the provided documents do not ..."
2,Which California community college has the hig...,"Using the most recent IPEDS data, EAST LOS ANG...","I am sorry, but the provided documents do not ..."
3,Which California community college has the hig...,"Using the most recent IPEDS data, EVERGREEN VA...","As of Fall 2006, salaries for tenured and tenu..."
4,Which California community college has the low...,"Using the most recent IPEDS data, SHASTA COLLE...","I am sorry, but the provided documents do not ..."


In [45]:
rouge_eval_task = EvalTask(dataset=vai_eval_dataset_3,
                           metrics=["rouge_l_sum"],
                           ).evaluate(experiment_run_name="test-1-run-10",)




Associating projects/1062597788108/locations/us-central1/metadataStores/default/contexts/test-1-test-1-run-10 to Experiment: test-1


Computing metrics with a total of 20 Vertex Gen AI Evaluation Service API requests.


100%|██████████| 20/20 [00:02<00:00,  9.55it/s]

All 20 metric requests are successfully computed.
Evaluation Took:2.101318584056571 seconds


In [48]:
# Examine results
print(rouge_eval_task.summary_metrics)
rouge_eval_task.metrics_table




{'row_count': 20, 'rouge_l_sum/mean': np.float64(0.23526377530000003), 'rouge_l_sum/std': np.float64(0.1314971933017686)}


,instruction,reference,response,rouge_l_sum/score
0,Which California community college awarded the...,"Using the most recent IPEDS data, WEST LOS ANG...","I am sorry, but based on the provided document...",0.222222
1,Which California community college has the hig...,"Using the most recent IPEDS data, MT SAN ANTON...","I am sorry, but the provided documents do not ...",0.228571
2,Which California community college has the hig...,"Using the most recent IPEDS data, EAST LOS ANG...","I am sorry, but the provided documents do not ...",0.089552
3,Which California community college has the hig...,"Using the most recent IPEDS data, EVERGREEN VA...","As of Fall 2006, salaries for tenured and tenu...",0.109589
4,Which California community college has the low...,"Using the most recent IPEDS data, SHASTA COLLE...","I am sorry, but the provided documents do not ...",0.315789
5,Which California community college has the hig...,"Using the most recent IPEDS data, SANTA MONICA...","I am sorry, but the provided documents do not ...",0.260870
6,Which California community college has the low...,"Using the most recent IPEDS data, SOLANO COMMU...",California community colleges usually charge a...,0.150000
7,Which California community college has the mos...,"Using the most recent IPEDS data, SOUTHWESTERN...","I am sorry, but the provided documents do not ...",0.115385
8,Which California community college had the mos...,"Using the most recent IPEDS data, ORANGE COAST...","I am sorry, but the provided documents do not ...",0.101695
9,How many California community college undergra...,"Using the most recent IPEDS data, 1491.0 under...","I'm sorry, but the provided documents do not c...",0.313043


In [42]:
# Examine some possible metric names
MetricPromptTemplateExamples.list_example_metric_names()


#
# eval_result = EvalTask(dataset=vai_eval_dataset[:5],
#                        metrics=["pairwise_question_answering_quality"],
#                        experiment="test-1",
#                        ).evaluate(experiment_run_name="test-1-run-9",)


['coherence',
 'fluency',
 'safety',
 'groundedness',
 'instruction_following',
 'verbosity',
 'text_quality',
 'summarization_quality',
 'question_answering_quality',
 'multi_turn_chat_quality',
 'multi_turn_safety',
 'pairwise_coherence',
 'pairwise_fluency',
 'pairwise_safety',
 'pairwise_groundedness',
 'pairwise_instruction_following',
 'pairwise_verbosity',
 'pairwise_text_quality',
 'pairwise_summarization_quality',
 'pairwise_question_answering_quality',
 'pairwise_multi_turn_chat_quality',
 'pairwise_multi_turn_safety']

In [24]:
vai_eval_dataset
dir(eval_result)

eval_result.metrics_table
eval_result.summary_metrics


{'row_count': 5,
 'pairwise_question_answering_quality/candidate_model_win_rate': np.float64(0.8),
 'pairwise_question_answering_quality/baseline_model_win_rate': np.float64(0.2)}

In [43]:
evaluator = EvalTask()

dir(evaluator)



TypeError: EvalTask.__init__() missing 2 required keyword-only arguments: 'dataset' and 'metrics'